# Train sp1DLC

Choose **Runtime > Change runtime type > GPU**, then run the four code cells in order. The prepared project contains all 1,865 images, the 31 updated label tables, and the config. The prepare cell handles the transfer files automatically. The new training run and checkpoints stay in your Google Drive. No original video files are needed for training.


In [ ]:
%pip -q install --pre deeplabcut


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime, timezone
import zipfile, shutil, yaml, torch, deeplabcut

assert torch.cuda.is_available(), 'Select a GPU runtime, then rerun this cell.'
MY_DRIVE = Path('/content/drive/MyDrive')
EXPORT = MY_DRIVE / 'sp1DLC' / 'MobileDLC_Colab_Export_2026-09-24'
PARTS = [EXPORT / f'sp1DLC_colab_ready_20260924.zip.part{i:02d}' for i in range(1, 10)]
missing = [part.name for part in PARTS if not part.is_file()]
assert not missing, f'Missing Drive transfer files: {missing}'
ZIP = Path('/content/sp1DLC_colab_ready_20260924.zip')
with ZIP.open('wb') as output:
    for part in PARTS:
        with part.open('rb') as source:
            shutil.copyfileobj(source, output)
assert ZIP.stat().st_size == 169245069, 'Project transfer was incomplete; rerun this cell.'
RUNS = MY_DRIVE / 'sp1DLC' / 'Colab_Training_Runs'
RUN = RUNS / ('mobile_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S'))
PROJECT = RUN / 'sp1DLC'
CONFIG = PROJECT / 'config.yaml'

with zipfile.ZipFile(ZIP) as archive:
    files = [name for name in archive.namelist() if not name.endswith('/')]
    assert len(files) == 1928, f'Unexpected archive file count: {len(files)}'
    assert all(name.startswith('sp1DLC/') and '..' not in Path(name).parts for name in files), 'Unexpected ZIP path'
    assert archive.testzip() is None, 'Project transfer failed its integrity check'
    RUN.mkdir(parents=True, exist_ok=False)
    archive.extractall(RUN)

with CONFIG.open(encoding='utf-8') as handle:
    config = yaml.safe_load(handle)
config['project_path'] = str(PROJECT)
config['video_sets'] = {str(PROJECT / 'videos' / Path(str(old).replace('\\', '/')).name): value for old, value in config['video_sets'].items()}
with CONFIG.open('w', encoding='utf-8') as handle:
    yaml.safe_dump(config, handle, sort_keys=False)

images = list((PROJECT / 'labeled-data').glob('Test */*.png'))
tables = list((PROJECT / 'labeled-data').glob('Test */CollectedData_Kobe.h5'))
assert len(images) == 1865 and len(tables) == 31, (len(images), len(tables))
print('Ready:', len(images), 'images and', len(tables), 'label tables')
print('GPU:', torch.cuda.get_device_name(0))
print('Training project:', PROJECT)


In [ ]:
deeplabcut.create_training_dataset(str(CONFIG), Shuffles=[1], net_type='resnet_50', engine=deeplabcut.Engine.PYTORCH)
deeplabcut.train_network(str(CONFIG), shuffle=1, epochs=200, batch_size=8, save_epochs=10)
print('Training finished. Checkpoints are in:', PROJECT / 'dlc-models-pytorch')


In [ ]:
deeplabcut.evaluate_network(str(CONFIG), snapshotindex='all', per_keypoint_evaluation=True, plotting=False)
results = sorted((PROJECT / 'evaluation-results-pytorch').rglob('CombinedEvaluation-results.csv'))
print('Evaluation saved in:', PROJECT / 'evaluation-results-pytorch')
if results:
    import pandas as pd
    display(pd.read_csv(results[-1]))


The new run folder printed above is permanent in Drive. If Colab disconnects during training, do not rerun the prepare cell and create a second project accidentally; use the existing run folder to resume. This export is frozen to September 24, 2026.
